In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, f1_score, recall_score, balanced_accuracy_score, confusion_matrix


In [4]:
df = pd.read_csv('../data/train.csv')
df.head()

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


In [6]:
print("Shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

print("\nColumn names:")
print(df.columns)

print("\nTarget distribution:")
print(df["Irrigation_Need"].value_counts())

Shape: (630000, 21)

First 5 rows:


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low



Column names:
Index(['id', 'Soil_Type', 'Soil_pH', 'Soil_Moisture', 'Organic_Carbon',
       'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm',
       'Sunlight_Hours', 'Wind_Speed_kmh', 'Crop_Type', 'Crop_Growth_Stage',
       'Season', 'Irrigation_Type', 'Water_Source', 'Field_Area_hectare',
       'Mulching_Used', 'Previous_Irrigation_mm', 'Region', 'Irrigation_Need'],
      dtype='str')

Target distribution:
Irrigation_Need
Low       369917
Medium    239074
High       21009
Name: count, dtype: int64


In [7]:
df = df.drop(columns=['id'])

In [8]:
target_col = "Irrigation_Need"

X = df.drop(columns=[target_col])
y = df[target_col]

In [9]:
X = pd.get_dummies(X, drop_first=True)
print("\nEncoded feature shape:", X.shape)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)


Encoded feature shape: (630000, 35)


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("\nTrain shape:", X_train.shape)
print("Test shape:", X_test.shape)


Train shape: (504000, 35)
Test shape: (126000, 35)


In [11]:
gnb = GaussianNB()
gnb.fit(X_train, y_train)

,"priors priors: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"var_smoothing var_smoothing: float, default=1e-9Portion of the largest variance of all features that is added tovariances for calculation stability... versionadded:: 0.20",1e-09


In [12]:
y_proba = gnb.predict_proba(X_test)

y_pred_baseline = gnb.predict(X_test)

print("\n=== Baseline Classification Report (Multiclass) ===")
print(classification_report(
    y_test,
    y_pred_baseline,
    target_names=label_encoder.classes_
))


=== Baseline Classification Report (Multiclass) ===
              precision    recall  f1-score   support

        High       0.62      0.78      0.69      4202
         Low       0.85      0.84      0.85     73983
      Medium       0.74      0.73      0.73     47815

    accuracy                           0.80    126000
   macro avg       0.74      0.78      0.76    126000
weighted avg       0.80      0.80      0.80    126000



In [16]:
focus_class_name = "High"
focus_class_index = list(label_encoder.classes_).index(focus_class_name)

print(f"\nFocused class: {focus_class_name}")
print(f"Focused class index: {focus_class_index}")


Focused class: High
Focused class index: 0


In [17]:
# 1 = High, 0 = Not High
y_test_binary = (y_test == focus_class_index).astype(int)
baseline_binary_pred = (y_pred_baseline == focus_class_index).astype(int)

In [18]:
baseline_f1 = f1_score(y_test_binary, baseline_binary_pred)
baseline_recall = recall_score(y_test_binary, baseline_binary_pred)
baseline_bal_acc = balanced_accuracy_score(y_test_binary, baseline_binary_pred)

print("\n=== Baseline Metrics for Focused Class (High vs Not High) ===")
print("F1:", round(baseline_f1, 4))
print("Recall:", round(baseline_recall, 4))
print("Balanced Accuracy:", round(baseline_bal_acc, 4))

print("\nBaseline binary classification report:")
print(classification_report(
    y_test_binary,
    baseline_binary_pred,
    target_names=[f"Not {focus_class_name}", focus_class_name]
))


=== Baseline Metrics for Focused Class (High vs Not High) ===
F1: 0.6929
Recall: 0.7787
Balanced Accuracy: 0.8813

Baseline binary classification report:
              precision    recall  f1-score   support

    Not High       0.99      0.98      0.99    121798
        High       0.62      0.78      0.69      4202

    accuracy                           0.98    126000
   macro avg       0.81      0.88      0.84    126000
weighted avg       0.98      0.98      0.98    126000



In [19]:
threshold = 0.30
focused_class_proba = y_proba[:, focus_class_index]

threshold_pred = (focused_class_proba >= threshold).astype(int)

In [20]:
# evaluate
threshold_f1 = f1_score(y_test_binary, threshold_pred)
threshold_recall = recall_score(y_test_binary, threshold_pred)
threshold_bal_acc = balanced_accuracy_score(y_test_binary, threshold_pred)

print(f"\n=== Threshold-Based Metrics for {focus_class_name} (threshold={threshold}) ===")
print("F1:", round(threshold_f1, 4))
print("Recall:", round(threshold_recall, 4))
print("Balanced Accuracy:", round(threshold_bal_acc, 4))

print("\nThreshold-based binary classification report:")
print(classification_report(
    y_test_binary,
    threshold_pred,
    target_names=[f"Not {focus_class_name}", focus_class_name]
))



=== Threshold-Based Metrics for High (threshold=0.3) ===
F1: 0.6132
Recall: 0.8522
Balanced Accuracy: 0.9101

Threshold-based binary classification report:
              precision    recall  f1-score   support

    Not High       0.99      0.97      0.98    121798
        High       0.48      0.85      0.61      4202

    accuracy                           0.96    126000
   macro avg       0.74      0.91      0.80    126000
weighted avg       0.98      0.96      0.97    126000



In [21]:
print("\nBaseline confusion matrix:")
print(confusion_matrix(y_test_binary, baseline_binary_pred))

print("\nThreshold confusion matrix:")
print(confusion_matrix(y_test_binary, threshold_pred))


Baseline confusion matrix:
[[119828   1970]
 [   930   3272]]

Threshold confusion matrix:
[[117902   3896]
 [   621   3581]]


In [22]:
comparison = pd.DataFrame({
    "Model Version": ["Baseline", f"Threshold = {threshold}"],
    "F1": [baseline_f1, threshold_f1],
    "Recall": [baseline_recall, threshold_recall],
    "Balanced Accuracy": [baseline_bal_acc, threshold_bal_acc]
})

print("\n=== Metric Comparison ===")
display(comparison)


=== Metric Comparison ===


,Model Version,F1,Recall,Balanced Accuracy
0,Baseline,0.692927,0.778677,0.881251
1,Threshold = 0.3,0.613237,0.852213,0.910113


In [23]:
thresholds = [0.20, 0.30, 0.40, 0.50, 0.60]
results = []

for t in thresholds:
    pred_t = (focused_class_proba >= t).astype(int)
    results.append({
        "Threshold": t,
        "F1": f1_score(y_test_binary, pred_t),
        "Recall": recall_score(y_test_binary, pred_t),
        "Balanced Accuracy": balanced_accuracy_score(y_test_binary, pred_t)
    })

threshold_results = pd.DataFrame(results)

print("\n=== Threshold Tuning Results ===")
display(threshold_results.sort_values(by="F1", ascending=False))


=== Threshold Tuning Results ===


,Threshold,F1,Recall,Balanced Accuracy
4,0.6,0.720453,0.726797,0.858382
3,0.5,0.695986,0.775821,0.880086
2,0.4,0.662622,0.818658,0.898077
1,0.3,0.613237,0.852213,0.910113
0,0.2,0.538896,0.874584,0.913638


## Discussion

I chose to focus on the **High** irrigation need class because it is the most important class to detect correctly in practice. If a truly high-irrigation case is missed, that could lead to under-watering and negatively affect crop growth or yield.

For my focused evaluation metric, I chose **recall**. Recall is especially useful here because it measures how many of the actual **High** cases were correctly identified by the model. Since the cost of missing a High-irrigation case is likely more serious than making an extra false alarm, recall is a reasonable metric to prioritize.

Using the **baseline** Naive Bayes predictions, the recall for the **High** class was **0.7787**. After applying a custom threshold of **0.30** to the predicted probability for the **High** class, the recall increased to **0.8522**. This means the threshold-based model identified a larger share of the actual High-irrigation cases than the default model did.

The tradeoff is that lowering the threshold made the model more likely to predict **High**, which increased the number of **false positives**. This can be seen in the classification report and confusion matrix --> recall improved, but **precision** dropped from **0.62** to **0.48**, and **F1** decreased from **0.6929** to **0.6132**. So, the model became better at catching High-irrigation cases, but it also labeled more non-High cases as High. 

This demonstrates the following tradeoffs:
- A **lower threshold** increases sensitivity to the chosen class
- This usually improves **recall**
- But often reduces **precision** because of more false positives

Compared with the default Naive Bayes model, the threshold-tuned version gave me more control over how the model treated the **High** class. Although the threshold did not improve every metric, it did improve the metric I prioritized: **recall**. For this assignment, that makes the thresholded model more useful when the goal is to avoid missing important High-irrigation cases.